# 01. Загрузка и проверка данных

Ноутбук загружает один Excel/CSV-файл транзакций из `data/raw/`, нормализует названия колонок, выполняет проверки качества и собирает дневную витрину `sales_date × stock_code × market_id`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_FILE_CANDIDATES, INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from src.data_checks import (
    find_raw_data_file,
    normalize_column_names,
    run_all_data_checks,
)

## Загрузка файла

Поддерживаются `online_retail_II.xlsx`, `online_retail.xlsx`, `online_retail_II.csv`, `online_retail.csv`.

In [2]:
raw_path = find_raw_data_file(RAW_FILE_CANDIDATES)

if raw_path.suffix.lower() in ['.xlsx', '.xls']:
    sheets = pd.read_excel(raw_path, sheet_name=None)
    raw_transactions = pd.concat(sheets.values(), ignore_index=True)
elif raw_path.suffix.lower() == '.csv':
    raw_transactions = pd.read_csv(raw_path)
else:
    raise ValueError('Поддерживаются только Excel и CSV.')

transactions = normalize_column_names(raw_transactions)
transactions.head()

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Подготовка полей

Возвратные и отмененные операции не удаляются. Они помечаются отдельным флагом и учитываются в `net_sales_qty`.

In [3]:
transactions['invoice'] = transactions['invoice'].astype(str)
transactions['stock_code'] = transactions['stock_code'].astype('string').str.strip()
transactions['description'] = transactions['description'].astype('string').str.strip()
transactions['country'] = transactions['country'].astype('string').str.strip()
transactions['quantity'] = pd.to_numeric(transactions['quantity'], errors='coerce')
transactions['unit_price'] = pd.to_numeric(transactions['unit_price'], errors='coerce')
transactions['invoice_date'] = pd.to_datetime(transactions['invoice_date'], errors='coerce')

transactions['sales_date'] = transactions['invoice_date'].dt.date
transactions['market_id'] = transactions['country']
transactions['is_return_flag'] = transactions['invoice'].str.startswith('C', na=False) | (transactions['quantity'] < 0)
transactions['is_price_missing'] = transactions['unit_price'].isna()
transactions['is_price_invalid'] = transactions['unit_price'].isna() | (transactions['unit_price'] <= 0)
transactions['revenue'] = transactions['quantity'] * transactions['unit_price']

transactions.head()

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,sales_date,market_id,is_return_flag,is_price_missing,is_price_invalid,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,United Kingdom,False,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,United Kingdom,False,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,United Kingdom,False,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,United Kingdom,False,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,United Kingdom,False,False,False,30.0


## Проверки качества

In [4]:
quality_checks = run_all_data_checks(transactions)
quality_checks

,check,passed,metric,message
0,required_columns,True,{'missing_columns': []},Все обязательные колонки найдены.
1,negative_or_zero_prices,False,"{'invalid_price_rows': 6207, 'missing_price_ro...",Есть строки с нулевой или отрицательной ценой.
2,duplicate_transaction_rows,False,"{'duplicates_count': 34335, 'duplicates_share'...",Найдены полные дубли строк транзакций.
3,missing_invoice_date,True,"{'missing_invoice_date_count': 0, 'missing_inv...",Пропуски в дате счета не найдены.
4,missing_stock_code,True,"{'missing_stock_code_count': 0, 'missing_stock...",Пропуски кода товара не найдены.
5,missing_prices_share,True,"{'missing_prices_count': 0, 'missing_prices_sh...",Пропуски цены не найдены.
6,returns_share,True,"{'returns_count': 22951, 'returns_share': 0.02...",Возвраты и отмененные счета посчитаны отдельно...


## Сохранение подготовленной таблицы

In [5]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
prepared_columns = [
    'invoice',
    'stock_code',
    'description',
    'quantity',
    'invoice_date',
    'sales_date',
    'unit_price',
    'customer_id',
    'country',
    'market_id',
    'is_return_flag',
    'is_price_missing',
    'is_price_invalid',
    'revenue',
]
transactions_clean = transactions[prepared_columns].copy()
transactions_clean.to_csv(INTERIM_DATA_DIR / 'transactions_clean.csv', index=False)
quality_checks.to_csv(INTERIM_DATA_DIR / 'data_quality_checks.csv', index=False)
transactions_clean.head()

,invoice,stock_code,description,quantity,invoice_date,sales_date,unit_price,customer_id,country,market_id,is_return_flag,is_price_missing,is_price_invalid,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,2009-12-01,6.95,13085.0,United Kingdom,United Kingdom,False,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,2009-12-01,6.75,13085.0,United Kingdom,United Kingdom,False,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,2009-12-01,6.75,13085.0,United Kingdom,United Kingdom,False,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2009-12-01,2.10,13085.0,United Kingdom,United Kingdom,False,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,2009-12-01,1.25,13085.0,United Kingdom,United Kingdom,False,False,False,30.0


## Дневная витрина продаж

Целевой уровень: одна строка = `sales_date × stock_code × market_id`.

In [8]:
mart_source = transactions_clean.dropna(
    subset=['sales_date', 'stock_code', 'market_id']
).copy()

mart_source['sales_qty_for_sum'] = mart_source['quantity'].where(
    ~mart_source['is_return_flag'],
    0
)

mart_source['returns_qty_for_sum'] = mart_source['quantity'].where(
    mart_source['is_return_flag'],
    0
)

mart_source['unit_price_positive'] = mart_source['unit_price'].where(
    mart_source['unit_price'] > 0
)

mart_daily_sales = (
    mart_source
    .groupby(['sales_date', 'stock_code', 'market_id'], as_index=False, sort=False)
    .agg(
        description=('description', 'last'),
        sales_qty=('sales_qty_for_sum', 'sum'),
        avg_unit_price=('unit_price_positive', 'mean'),
        revenue=('revenue', 'sum'),
        invoices_cnt=('invoice', 'nunique'),
        customers_cnt=('customer_id', 'nunique'),
        returns_qty=('returns_qty_for_sum', 'sum'),
        net_sales_qty=('quantity', 'sum'),
        is_return_flag=('is_return_flag', 'max'),
        is_price_missing=('is_price_missing', 'max'),
    )
)

mart_daily_sales['returns_qty'] = mart_daily_sales['returns_qty'].abs()

mart_daily_sales['sales_date'] = pd.to_datetime(mart_daily_sales['sales_date'])
mart_daily_sales['weekday'] = mart_daily_sales['sales_date'].dt.dayofweek
mart_daily_sales['month'] = mart_daily_sales['sales_date'].dt.month
mart_daily_sales['year'] = mart_daily_sales['sales_date'].dt.year
mart_daily_sales['is_weekend'] = mart_daily_sales['weekday'].isin([5, 6])

mart_daily_sales = mart_daily_sales[
    [
        'sales_date',
        'stock_code',
        'description',
        'market_id',
        'sales_qty',
        'avg_unit_price',
        'revenue',
        'invoices_cnt',
        'customers_cnt',
        'returns_qty',
        'net_sales_qty',
        'weekday',
        'month',
        'year',
        'is_weekend',
        'is_return_flag',
        'is_price_missing',
    ]
]

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
mart_daily_sales.to_csv(PROCESSED_DATA_DIR / 'mart_daily_sales.csv', index=False)

mart_daily_sales.head()

,sales_date,stock_code,description,market_id,sales_qty,avg_unit_price,revenue,invoices_cnt,customers_cnt,returns_qty,net_sales_qty,weekday,month,year,is_weekend,is_return_flag,is_price_missing
0,2009-12-01,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,United Kingdom,46,9.430000,335.05,10,8,3,43,1,12,2009,False,True,False
1,2009-12-01,79323P,PINK CHERRY LIGHTS,United Kingdom,103,6.378571,588.15,7,7,2,101,1,12,2009,False,True,False
2,2009-12-01,79323W,WHITE CHERRY LIGHTS,United Kingdom,93,7.235000,565.72,12,10,1,92,1,12,2009,False,True,False
3,2009-12-01,22041,"RECORD FRAME 7"" SINGLE SIZE",United Kingdom,96,2.100000,201.60,2,2,0,96,1,12,2009,False,False,False
4,2009-12-01,21232,STRAWBERRY CERAMIC TRINKET BOX,United Kingdom,183,1.370000,244.59,11,10,0,183,1,12,2009,False,False,False


## Что перенести в отчет

- число строк в исходном файле: `[A]`;
- доля возвратов: `[B]`;
- доля строк без цены: `[C]`;
- число строк в дневной витрине: `[D]`.